# Showcase B (ParaView + trame) inside Jupyter

Extra E8. The Wavelet -> Contour pipeline is the same `ParaViewPipeline` class that `showcase-b/app.py` uses (imported, not copied). Rendering is done by ParaView in this notebook's kernel; the browser shows the image stream through a trame view.

**Setup (once, from the repo root)**
```bash
python3 -m venv .jvenv
.jvenv/bin/pip install -r setup/requirements-jupyter.txt
scripts/install_jupyter_kernel.sh      # registers kernel "ParaView 6.2 pvpython" into .jvenv
.jvenv/bin/jupyter lab --port 8888      # open notebooks/trame_in_jupyter.ipynb, pick that kernel
```
Why this way: the kernel is started as `pvpython --venv .jvenv -m ipykernel_launcher -f {connection_file}`, so `import paraview` works with ParaView's own VTK and `.jvenv` only supplies pure Python packages (jupyterlab, ipykernel, trame, trame-vtk, trame-vuetify, trame-jupyter-extension). No `vtk` wheel is installed. `trame-jupyter-extension` tunnels the trame websocket over the kernel's comm channel, so no extra port is needed.

In [ ]:
import os, sys
# kernelspec already sets these; repeated here so the cell also works under a different kernel launcher
os.environ.setdefault("TRAME_BACKEND", "jupyter")
os.environ.setdefault("TRAME_IFRAME_BUILDER", "jupyter-extension")
sys.path.insert(0, os.path.abspath("../showcase-b"))
from app import ParaViewPipeline   # importing app.py does not start a server (main() is guarded)
pipeline = ParaViewPipeline(size=64)   # 64^3 Wavelet -> Contour(RTData=157), shown in the default RenderView
pipeline.stats()

trame server, one VtkRemoteView of the ParaView render view, and an iso slider that calls `pipeline.set_iso`.

In [ ]:
from trame.app import get_server
from trame.ui.vuetify3 import SinglePageLayout
from trame.widgets import vuetify3 as v3, paraview as pv_widgets
from trame_vtk.modules.vtk.serializers.data import merge_to_polydata_serializer
from trame_vtk.modules.vtk.serializers.registry import register_instance_serializer
register_instance_serializer("vtkPartitionedDataSetCollection", merge_to_polydata_serializer)  # as in app.py

server = get_server("e8_jupyter", client_type="vue3")
state, ctrl = server.state, server.controller
state.iso = pipeline.iso

@state.change("iso")
def on_iso(iso, **_):
    pipeline.set_iso(float(iso))
    ctrl.view_update()

with SinglePageLayout(server) as layout:
    layout.title.set_text("Showcase B in Jupyter")
    with layout.toolbar:
        v3.VSpacer()
        v3.VSlider(v_model=("iso",), min=40, max=270, step=1, label="Iso value",
                   hide_details=True, density="compact", style="max-width: 320px;")
        v3.VLabel("{{ Math.round(iso) }}", classes="mx-2")
    with layout.content:
        with v3.VContainer(fluid=True, classes="pa-0 fill-height"):
            view = pv_widgets.VtkRemoteView(pipeline.view, ref="view", interactive_ratio=1)
            ctrl.view_update = view.update

In [ ]:
await layout.ready
await layout.display_cell(height='480px')

Change the slider above: the contour is recomputed by ParaView in the kernel and a new frame is streamed. The same value can be set from Python, e.g. `state.iso = 200; state.flush()`.